<a href="https://colab.research.google.com/github/kylashrao/DataScience-Projects/blob/main/Project%20(Binary%20Classification%20with%20PyTorch).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PyTorch is an open-source deep learning framework developed by Meta. It is widely praised for its dynamic computation graphs (define-by-run), pythonic feel, and deep integration with the standard Python data science ecosystem (NumPy, Pandas, SciPy, and Scikit-learn).

Part 1: Core Concepts of PyTorch
Key Abstractions
Tensors (torch.Tensor): The fundamental data structure in PyTorch, similar to NumPy arrays, but optimized to run seamlessly on GPUs for high-speed parallel computation.

Autograd (torch.autograd): PyTorch's automatic differentiation engine. It tracks operations on tensors to compute gradients automatically during backpropagation.

torch.nn (Neural Network Modules): Built-in classes and layers (like nn.Linear, nn.ReLU) to simplify building architectures.

torch.utils.data (DataLoaders & Datasets): Tools used to efficiently handle mini-batching, shuffling, and parallel data loading during training.

Part 2: End-to-End Project (Binary Classification with PyTorch)
We will build, train, and evaluate a Deep Neural Network (DNN) using PyTorch to perform binary classification on the breast cancer dataset, preprocessing features with Scikit-learn and Pandas.

Prerequisite: Ensure PyTorch is installed (pip install torch).

1. Import Libraries & Load Data

In [2]:
pip install torch

In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Split into Training (80%) and Testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

2. Feature Scaling & Tensor Conversion
Neural networks require scaled inputs for stable gradient descent. Once scaled, we convert our NumPy arrays into PyTorch Tensors and wrap them in a DataLoader for mini-batch processing.

In [4]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch Tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Create DataLoader for batching
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

3. Define the Neural Network Architecture
In PyTorch, custom models are built by subclassing nn.Module. We define our layers in __init__ and map the data flow in the forward method.

In [5]:
class PyTorchDNN(nn.Module):

  def __init__(self, input_dim):
    super(PyTorchDNN, self).__init__()
    self.layer1 = nn.Linear(input_dim, 16)
    self.relu = nn.ReLU()
    self.dropout = nn.Dropout(0.2)
    self.layer2 = nn.Linear(16, 8)
    self.output = nn.Linear(8, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.relu(self.layer1(x))
    x = self.dropout(x)
    x = self.relu(self.layer2(x))
    x = self.sigmoid(self.output(x))
    return x


# Instantiate model, loss function, and optimizer
input_dim = X_train.shape[1]
model = PyTorchDNN(input_dim)
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.001)

4. Train the PyTorch Model
We write an explicit training loop where gradients are zeroed out, losses are computed via forward passes, backpropagated via .backward(), and weights are updated using the optimizer.

In [6]:
epochs = 50

for epoch in range(epochs):
  model.train()  # Set model to training mode
  running_loss = 0.0

  for batch_X, batch_y in train_loader:
    # 1. Zero the gradients
    optimizer.zero_grad()

    # 2. Forward pass
    predictions = model(batch_X)

    # 3. Compute loss
    loss = criterion(predictions, batch_y)

    # 4. Backward pass (compute gradients)
    loss.backward()

    # 5. Update weights
    optimizer.step()

    running_loss += loss.item()

  if (epoch + 1) % 10 == 0:
    print(
        f'Epoch [{epoch + 1}/{epochs}], Loss: {running_loss / len(train_loader):.4f}'
    )

Epoch [10/50], Loss: 0.1010
Epoch [20/50], Loss: 0.0591
Epoch [30/50], Loss: 0.0457
Epoch [40/50], Loss: 0.0384
Epoch [50/50], Loss: 0.0325


5. Model Evaluation
Evaluate how well the trained PyTorch network generalizes to unseen test data.

In [7]:
model.eval()  # Set model to evaluation mode
with torch.no_grad():
  # Forward pass on test data
  y_pred_prob = model(X_test_tensor)
  # Convert probabilities to binary classes (0 or 1)
  y_pred = (y_pred_prob > 0.5).float().numpy()

# Metrics
print('\n--- PyTorch Deep Learning Performance ---')
print(f'Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=data.target_names))


--- PyTorch Deep Learning Performance ---
Test Accuracy: 0.9825

Confusion Matrix:
[[42  1]
 [ 1 70]]

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        43
      benign       0.99      0.99      0.99        71

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

